<!-- dd:dd-lesson-eo-3 -->

# Repeat and deep-learning patterns

*Einops · `eo-3`*

Work through this with the **Delta Drills** side panel open. It picks what you practise, sends you to the cell, and records how it went — you do not need to read this notebook in order.


In [ ]:
# === Delta Drills ===
# Which lesson this notebook is, for the side panel. Nothing to run.
DD_LESSON_ID = "eo-3"


In [ ]:
#@title 🔧 Delta Drills checker — run me first { display-mode: "form" }
# Delta Drills — problem checker. Generated; see scripts/colab_grader.py.
#
# `dd_check(<problem id>)` runs your `solve` against the same cases the tutor
# grades with, and tells you which ones failed. It reads `solve` out of the
# notebook, so define it (run your cell) before you check.
import base64
import json
import sys
import zlib

import numpy as np

# Filled in by the generated cell that follows this source: {qid: {fn, cases}}.
_DD_TESTS = {}

# Where the ARENA digits fixture is fetched from, also filled in by that cell.
_DD_FIXTURE_URL = ""
_DD_FIXTURE_PATH = "/delta_numbers.npy"

_DD_RTOL = 1e-5
_DD_ATOL = 1e-6


def _dd_install_fixtures():
    """Make `np.load('/delta_numbers.npy')` work here the way it does in the app.

    24 of the einops drills are written against the ARENA digits image, and the
    bank refers to it by an absolute path the backend rewrites at grade time
    (`code_runner.CODE_PREAMBLE`). Nothing rewrote it in a notebook, so those
    problems could not run at all in Colab — not the checker, not the starter
    code the learner was sent there to fill in. Downloaded on first use, so the
    six notebooks that never touch it never pay for it.
    """
    import os
    import urllib.request

    original = np.load
    if getattr(original, "_dd_patched", False):
        return

    def _load(file, *args, **kwargs):
        if str(file) == _DD_FIXTURE_PATH and not os.path.exists(_DD_FIXTURE_PATH):
            if not _DD_FIXTURE_URL:
                raise FileNotFoundError(
                    "This drill needs the ARENA digits fixture and no source was "
                    "compiled into this notebook — regenerate it."
                )
            urllib.request.urlretrieve(_DD_FIXTURE_URL, _DD_FIXTURE_PATH)
        return original(file, *args, **kwargs)

    _load._dd_patched = True
    np.load = _load


def _dd_load(blob):
    """The test payload, deflated and base64'd.

    Not encryption and not pretending to be — it is one `zlib.decompress` away.
    It is compressed because the payload for a 84-problem notebook is ~80 KB of
    JSON, and out of sight because an expanded grader cell would otherwise sit
    in the notebook spelling out the expected answer to every problem below it.
    """
    return json.loads(zlib.decompress(base64.b64decode(blob)).decode("utf-8"))


def _dd_tensor(value):
    # torch only if something already imported it. numpy-only notebooks must
    # not pay a torch import to compare two lists of ints.
    torch = sys.modules.get("torch")
    return torch is not None and isinstance(value, torch.Tensor)


def _dd_close(a, b):
    """Tolerance compare, but ONLY when a float or complex is involved.

    torch defaults to float32 where numpy defaults to float64 and honest
    answers differ in reduction order, so exact equality fails correct work.
    Integer and boolean results stay exact — an index answer (argmax, nonzero,
    searchsorted) must never be fudged by a tolerance. Returns None to mean
    "not a float comparison, use exact equality".
    """
    try:
        floaty = any(
            np.issubdtype(x.dtype, np.floating) or np.issubdtype(x.dtype, np.complexfloating)
            for x in (a, b)
        )
        if not floaty:
            return None
        if a.shape != b.shape:
            return False
        return bool(np.allclose(a, b, rtol=_DD_RTOL, atol=_DD_ATOL, equal_nan=True))
    except Exception:
        return None


def _dd_array_equal(a, b):
    close = _dd_close(a, b)
    if close is not None:
        return close
    return bool(np.array_equal(a, b))


def _dd_equal(a, b):
    if _dd_tensor(a) or _dd_tensor(b):
        try:
            a2 = a.detach().cpu().numpy() if _dd_tensor(a) else np.asarray(a)
            b2 = b.detach().cpu().numpy() if _dd_tensor(b) else np.asarray(b)
            return _dd_array_equal(a2, b2)
        except Exception:
            # dtypes numpy cannot hold (bfloat16, conj views): equal tensors
            # must not grade as unequal — ask torch itself.
            torch = sys.modules.get("torch")
            if torch is not None and isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor):
                try:
                    return bool(torch.equal(a.detach().cpu().resolve_conj(),
                                            b.detach().cpu().resolve_conj()))
                except Exception:
                    return False
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        return _dd_array_equal(np.asarray(a), np.asarray(b))
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        if len(a) != len(b):
            return False
        return all(_dd_equal(x, y) for x, y in zip(a, b))
    close = _dd_close(np.asarray(a), np.asarray(b))
    if close is not None:
        return close
    return bool(a == b)


def _dd_seed():
    # The same seed before the actual-side and the expected-side setup runs, for
    # BOTH rngs: setup executes twice, so an unseeded torch.rand in a fixture
    # would hand the two sides different data and fail an honest answer.
    np.random.seed(0)
    torch = sys.modules.get("torch")
    if torch is not None:
        torch.manual_seed(0)


def _dd_show(value, limit=320):
    try:
        text = repr(value)
    except Exception as exc:
        text = "<unrepresentable: %s>" % type(exc).__name__
    text = " ".join(text.split()) if len(text) > limit else text
    if len(text) > limit:
        text = text[: limit - 1] + "…"
    return text


def dd_check(question_id, verbose=True):
    """Grade the `solve` you just defined against this problem's cases.

    Returns True when every case passes. Prints which ones did not, with the
    fixture, what was expected and what came back — a failing grade should be
    evidence you can act on, not a verdict.
    """
    qid = str(question_id)
    entry = _DD_TESTS.get(qid)
    if entry is None:
        print("No checker for problem %s in this notebook." % qid)
        return False

    # The learner's namespace, not this function's: `solve` lives in the cell
    # they ran, and in Colab that is the caller's globals.
    try:
        env = sys._getframe(1).f_globals
    except Exception:
        env = globals()

    fn_name = entry.get("fn") or "solve"
    if fn_name not in env:
        print("❌ `%s` is not defined yet — run your solution cell first." % fn_name)
        return False

    cases = entry.get("cases") or []
    failures = []
    for i, case in enumerate(cases, 1):
        # A fresh copy per case: fixtures are exec'd, and exec'ing them into the
        # notebook's own globals would quietly overwrite whatever the learner
        # named `x` two cells ago.
        ns = dict(env)
        try:
            if case.get("setup_code"):
                _dd_seed()
                exec(case["setup_code"], ns)
            actual = eval(case["call"], ns)
            expected_setup = case.get("expected_setup_code") or case.get("setup_code")
            if expected_setup:
                _dd_seed()
                exec(expected_setup, ns)
            expected = eval(case["expected_expr"], ns)
            if not _dd_equal(actual, expected):
                failures.append((i, case, _dd_show(expected), _dd_show(actual), ""))
        except Exception as exc:
            failures.append((i, case, "", "", "%s: %s" % (type(exc).__name__, exc)))

    total = len(cases)
    if not failures:
        print("✅ Problem %s — %d/%d cases passed." % (qid, total, total))
        return True

    print("❌ Problem %s — %d of %d cases failed." % (qid, len(failures), total))
    if verbose:
        for i, case, expected, actual, error in failures:
            print("\n  case %d" % i)
            if case.get("setup_code"):
                for line in case["setup_code"].strip().splitlines():
                    print("    given     %s" % line)
            print("    called    %s" % _dd_show_source(case.get("call", "")))
            if error:
                print("    raised    %s" % error)
            else:
                print("    expected  %s" % expected)
                print("    you got   %s" % actual)
    return False


def _dd_show_source(text, limit=160):
    text = " ".join(str(text).split())
    return text if len(text) <= limit else text[: limit - 1] + "…"

_DD_FIXTURE_URL = "https://raw.githubusercontent.com/AkiraTheSquid/arena-book-colab/main/ARENA_5.0/ch-1-foundations/numbers.npy"
_dd_install_fixtures()
_DD_TESTS = _dd_load(
    "eNrtXHuP4zYO/yrB/NPkkLrRw5a9QIH7EP1vdrCYm/HODDabzCWZ7vQW/e4nUaQeiZ/xo+31gMSJJcYUaYr6kZTz/UYwdfNh"
    "8f3m805/3Bz321/Lm/Xi5uH+WB51y+33m2N5env99LB/LA3Fy9fX/eG0OO0PD8+L++Pi9HGHTeXLbv96/Li7PxwWPy9Oyanc"
    "HfeH5e412e7vH5c//PRYbk/3n3ZvX/9VHo7J7vW3H1Yr8+snTa5/dLu5s5y3W8NoqbluyyWMaKmJVsnx+f61XK0XL7vT8pQc"
    "9m+7x7D78/b+pHkuV8lpr/s/a66nTK4W/1iwbLX4px7R/eF+9xRdUg+m3C71NR9Pv72WP/tfrVZmLOX7a/lwKh8/6S8HGNRS"
    "rBcy3awXLN3on6UZEykvCsVTrn918/t60V9hVgNueFz64bxpWfNVcihBdsOca/6rSE8vx5fd8XS/ewglW+vr/QI3QH/1zTVS"
    "/XJ4K9eL29tbIxjwEHf6vOlMDzJdL7L1QjWemdPbfL0o9I/NBZhpaDtnmgnTsjJp9NypBfgwzZIp/dYXZAVQdWjhmi83kukr"
    "ctGp5e5unFtthKi51ZqZlkysbj98YHdT3+/g9p59FTCMtP6rucvrRV7/NbqzNSegzzut0RuRsonc0df9r+Xjy9dlR8cEKviR"
    "/QUd1Ebr1B6MEIzlghVZJiQv5FgeSg9ZmGHXWq/xEpr5hI4K7BQmtDtwewBX0NYPs9o6MndI7aG1f9jkXzo1GhWCKs/VqNvS"
    "c6cvJnX6qZmKOPXdd6a/geNLz86shq/6lTB2qdsk0MZnUn8zS2uanp0N/GWmv2e6TQFtfKb0t1y35OnZ2YBfemcm8qHO7GF7"
    "DMGUXuMSo81ko7l81BZQvpp+0Wwb+iLaHgxtlYkEvW2W4pibO1v1vfvcsIItK5dC8HGCrxY/aawTzgMZyMxnknljZNskPDVH"
    "OCi445XtFoaYHgY9LIGD/UVlO/yCW/2ZHp7Awf6isr2vjgPjSc/shs2kw/QK04htHrURjZ7PZ/V4+9zdcsPxc13mf0LgcrTI"
    "pQm2HFtwy/Ea4HK8BrlkPrSy4IXznLFUqGKj8lTx/Pp19xihblp3GyEMxjqdUMyxZultt60Ao4TIpbFBBIikQ8MQvBL7aFKa"
    "+eQVoEWcA0ATLU2rPYxGtekUDpa0tkDsaF1ybuK6/q0mRufKARHRrxWab4UekIAIVJqgnm5dj1apR5SCZaQmHFP9WwHSWO1o"
    "DlnRrzXwfnn6/yxSvywSeLlsY7JIRaGKTGzERqUi24wXo1UGF+EUnTyucC8WvLh7YS7BvWTwSt0LzQ8ySvalglfuXphTcC9I"
    "LoRvFr7Hi+MYucUKlxgvKKKTPxxD37FWIOkVvMEluTfcBWMKwVtuwneEc0ZI0ESWKpPN+arL9Yz+0o7w9IXWiy816jI9nbC9"
    "R5dadPu9sr2PvcQGEmZ0a2MbbsObL+3h3DhiV4Q0dYHLpPSNAZCwyjftIoFDQ2A0Fr1dL02PhB6ZwMH+IjXtKbSnCRxs+6T0"
    "fU0viJ2i4OlupnkV8MTAKYyXegmy35XHpVmrjNNe4fjZDOOnUD7K5xRDfd+38uXp+dSU03lsvz94Ee1UqmQMejvlc0I/1+8e"
    "eWG8r8vaXB1gDiumnElM9EfBwXop66Dio+3xnsseMJJIFHqz8Bh4tuBgW62ri49XKTiwFhUYCptJg+pKuwiGXb2mzmzw4f13"
    "dypcgTbBqhBN/VzMA3sOHd3zoda9Ha6DPWjhowKfTknduQU/m+UXWVtrG9XwxOrFZWKdyka85oClPqU1/tDuXAfrMiWbqfky"
    "aKXnK5SCTSxF7AT8WVCQ5v+b8U5d7PM3inf6tvcq7IxX8Lki3unV3hjv9G2vi3d6tf9F45047Bkx3pGzjv+83lXfG3jJwflf"
    "SviiD8lDL8ld3rDbDdXXqlEI9nR1lBgxnLlJAIrOYUprwTAd7DfIkZvvyhzoWw9jsLoIdopkHXGUja06+NRxNBT5yKZz61VV"
    "6uIj7kKo+oZqh9V0bn+hUrwVEtubG8jI0UHjzW0+t9YQueML/1zpsJ2vdCbTdG4dZOQbL5zlEO9Jcy5yOoi+OuRZBhtRHCyH"
    "2KvX8PWgHr4sb0/Jf8rDXgvCyHuunWyu6W5OD7KpEg0dZjbUYb53cZfNUr5XCfjeJpt3jtVu0Du8rjfy/Sr4OK2AZ/gvclQR"
    "yqvCZlWOqh43od4u5nEf9QWGLsjWp9BNnOTqOcSaCGhmY/WTsBg8CY/lv2unIXNBzu653ePoK60Xu+cqIamrfTtE4E8dGPEB"
    "h4ihSZ+tIVbOILNT8A6JXpP/yUn+fBb5AwAunbjwkAH6YvLJVJxAKoZkDOk4XYv73BjlajjOZPuZ4WdOeR4iZETJiJQRLSNi"
    "7q7KHRRBvMQTQgyWCzIhHsSCOBADuj5dnq4e5LDt1ckfKby+QgaKOChioYiHIiaKuChio4hPkPtCAIdhHmyTtR8F6pqWfiRi"
    "SMWQjCEdJ/wtfEhC8JCnBJLs9e1nQXp2yTeiZETKiJYRMSdiLkJYaPkgG+SCTIgHsSAOxICuT5enq7uLKwdZ7eUVXl8hA0Uc"
    "FLFQxEMRE0VcFLFRxAfAIEwNjmbO0cw5mrlAMxeU+kY6gXQC6STSSZpbSCcz52HIyDkZOScjF2TkgmaEIGJBxIKIJRFLN9GI"
    "WGbuvnA0eI4Gz9HgBRq8wIkhkE4gnUA6iXSSZhnSySzxHBSxUMRDERNFXBSxUcRHESNFnBSxUsRLETOXEELT52j6HE1f0DTC"
    "KSKQTiCdQDpJXp/QGNJJ5e4PGT4nw+dk+MJNJ5olgogFEQsilkQsiVgSsVT+/uAk4DgJOE4CQROKojWkE0gnkE4inUQ6iXRS"
    "JZ6DIhaKeChiooiLIjaK+ChipIiTIlaKeCliptL+q2QMxbIVLYBi+gWwMqC6qHWNRdVfMfUA0MEkNh9MCjFhWOqTg0OzT8/l"
    "/eOxBRl2wbz2QpXIF7u6p7LC9JRPZV2RnvLieUBo9pe24kHZaQvcWEJv4lgtisminHUQ5AXBnAvbgvDsLKsfxHtBXBdEcLBx"
    "0WrZ9KgEDoa+MK2FHSHwd4ArM2cZEOWGKAeiIoEDBKJ2BMxhHriBHOjhoOzlYNuj5WCVwPplh0IrDrMokGWc/h5WbKeIQs+r"
    "5KivwbGZZyOv2GRQyDlqjVNtQK0JtSfeOTCpNJ2SQJHXCCr5Q0o/rRPsesEu9ypdk9C5LLzRGjLRsFWUuZHFOOlTr3Nut0tP"
    "kTVjqQmnpkqGdn7U5pqhZ0W/gZ/l4DkWE95v/WMAd5rsR73+jj9Yc9V+g/38tt264oDJmQIUmkCPcFW03YxPl/q3fxvBbKX0"
    "+Lp9ObVnHt+1DIayUizqawXVPvno9xPYjUA2A3G20yxMQl5ZJ+gEOTNXN/4DtFGzLaxpO5kvS4bVQlKiw5IRUMR/WSGwiHlC"
    "iwQxlvNokNKHhDjd/WnarOaQMybDwp0jETgOq5C0c9SiU0ycWfxrOx0GxmyaBbg44hjkxgj2yrpDdjZTxMVMETPPlMu5QS3h"
    "/Lm2EBTsIiEB2SwCnu2fD9ZtlQ6PrVtqLsys4m1AXl+jGsabjnYQT0BP2inYL6o6r6WIrNsSn00tFbNgN7PolpEnyCyuZfYj"
    "IyfAbJCc2XCacefAMhsEM/uRkbOC88z2Krq6sue57c2Bur8uQxgqManRjkUH6ivtGYReJAehODjtGOs3WCAUKWaCIk/684mb"
    "1XfdYQG21DVOCDu7F0PP8EhUDr3AItdBEca7YBHmwYjTh3nYdUZ91CCSOtTRiivs3ijuK54pnHbZAF+HG1qRAYARX9ADOMJE"
    "J3TVhJ+aMBL0g5J8LRfUxGQnjNSEgpqQDvSDonzdElTFQozFQCxfXAbBWFBcAsl8qdmuXwEeBcl8AQ8k414yDibgynkcjIAH"
    "VgDi+YIqCMi8hNwmhZivs6ZwGha/zF9EUT9IyL01cDAHV67iYBA8sAgYn6upMxgf8+Pj1iS4L7WncOr6QX++bgn6415/AvTn"
    "q5igPxFYBozP1ZEZjI/58XG7Z1T48nIKp64f9OfrcqA/7vUnQH++Sgf6E6xncaoRCmcIFWNPLWbzTG4KBbgRHZP/053g4ZJr"
    "IbH0kNgJymZzwexCvnOk7IHyuEkCWbkyw/3uFBI/1Une4+7yC6HFkDU365IZy/2K276PfrCco0b8revVyPF60zpl15lrMwyd"
    "kELL6nptrqALYmhcYUf3r53SDKPON1k98YZ70zTwpp2SCwPEavOdlU40nyfb8HH3rYN3+QQbFL7VRnOmrz2gswXmQQmHTolT"
    "qImgZGIGyahav/FFNbfFz5XXFUlvnaNNJWyopp4nbvNfUADHkvnQvAJ3WygyerRsLtXIoZkGhtuQvnWYpONYaVOJMah3q5me"
    "Gmt92KPmSY9u9XwZJFjiFEMW/KHAsEfARqwDDhU3KvHH24o2wX9d0C4aO305PqjktjhjAZ0ntNVGkmsLnhVTwfMUKnhaYsBT"
    "TnY6wDak6fTU8AjlxfnEjztNbgvhhGbZTPnEh/3+8NgJ0QNlDeywfeMXbGzLCCXOnjHOH6CVnv980R5I1AH9SPM1oVRd2INp"
    "srYQpbHkGNzlmmCtuRDaGgA1BCmRRdWEg82hW1t41RwCjfOYF4LKwE75XHbqna23Ino6ZUD8I3z8QyKxWUSqDX2CB0yLv8Eu"
    "kyY3/HffZXLVnw+17jJpqmTUuXW6N01Zqesda2NmaJDLH8EtzLuxJHJxVScjOPLgXwLm3lYSwDD7UFsw+63j+/2/DEIckw=="
)
print("Delta Drills checker ready — 23 problems. Run dd_check(<problem number>) under any of them.")


<!-- dd:dd-kp-einops-repeat-model -->

## einops.repeat — new axes and stretched axes

`einops.repeat-model`


The third einops function, **`einops.repeat`, is the mirror image of
reduce: the output may contain names (or factors) the input DOESN'T have**
— and the data is copied to fill them. Two distinct moves live under it:

**1. A brand-new axis** — a name on the right that's absent on the left:

> `repeat(cls, 'b d -> b t d', t=8)`
> — each (b, d) embedding is broadcast across 8 new time steps. This is
> einops' answer to `broadcast_to` / `unsqueeze` + `expand`: the classic use is
> spreading a class token or per-token weights across a new dimension.

**2. Stretching an existing axis** — a factor inside output parens that
wasn't in the input:

> `repeat(img, 'c h w -> c (h 3) w')` — each ROW appears 3× consecutively
> (h slow, 3 fast: row, its copies, next row). Nearest-neighbor upscaling
> is this on both axes: `'h w c -> (h 2) (w 2) c'`.
> `repeat(img, 'c h w -> c (3 h) w')` — the WHOLE image stacked 3 times
> (3 slow: full copy, then full copy). PyTorch's
> `repeat_interleave`-vs-`repeat` distinction, decided by paren ORDER
> instead of function choice.

The order rule is the one you already own from merging: **left = slow =
blocks, right = fast = within-block.** `(h k)` interleaves copies per row;
`(k h)` concatenates whole copies. Literal numbers can sit in the parens
directly, or be keywords (`(h k), k=3`).

New-axis literals work too: `'h w -> h w 3'` copies a grayscale image into
3 identical channels.


Task: broadcast an embedding across time; stretch rows; whole-image stack —
the three moves, distinguished.


In [ ]:
import torch as t
import einops

# 1. NEW AXIS: (b, d) -> (b, t, d), each embedding copied t times.
cls = t.tensor([[1.0, 2.0],
                [3.0, 4.0]])                # (b=2, d=2)
seq = einops.repeat(cls, 'b d -> b t d', t=3)
assert seq.shape == (2, 3, 2)
assert seq[0].tolist() == [[1.0, 2.0]] * 3   # identical copies down t

# 2. STRETCH, factor fast: each ROW repeats consecutively.
img = t.tensor([[1, 2],
                [3, 4]])                     # (h, w) for clarity
rows3 = einops.repeat(img, 'h w -> (h r) w', r=2)
assert rows3.tolist() == [[1, 2],
                          [1, 2],
                          [3, 4],
                          [3, 4]]            # row, its copy, next row

# 3. STRETCH, factor slow: the WHOLE block repeats.
whole = einops.repeat(img, 'h w -> (r h) w', r=2)
assert whole.tolist() == [[1, 2],
                          [3, 4],
                          [1, 2],
                          [3, 4]]            # full image, then again

# Nearest-neighbor 2x upscale: both axes, factor fast on each.
up = einops.repeat(img, 'h w -> (h a) (w b)', a=2, b=2)
assert up.tolist() == [[1, 1, 2, 2],
                       [1, 1, 2, 2],
                       [3, 3, 4, 4],
                       [3, 3, 4, 4]]
print("new axis:", tuple(cls.shape), "->", tuple(seq.shape),
      "| item 0 =", seq[0].tolist())
print("'(h r)' factor FAST — each row repeats:\n", rows3)
print("'(r h)' factor SLOW — the block repeats:\n", whole)
print("2x nearest-neighbor upscale:\n", up)




Why each step:

1. `(h r)` vs `(r h)` on the same data is the `repeat_interleave`-vs-`repeat` shootout
   resettled by one convention (left slow) instead of two function names —
   run both once, then trust the rule.
2. The new-axis case allocates real copies (unlike `expand`'s
   virtual stretch) — fine for drills; in memory-tight code you'd reach
   for broadcast_to semantics knowingly.
3. The upscale's per-axis factors (a, b) show the moves composing — each
   axis independently gets the "pixel becomes a block" treatment, and the
   2×2 blocks in the output are the proof.


<!-- dd:dd-q317 -->

### Problem 317 · faded — your turn

Triple an image's height by repeating each row consecutively.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
tensor([[[255, 255, 255,  ..., 255, 255, 255],
         [255, 255, 255,  ..., 255, 255, 255],
         [255, 255, 255,  ..., 255, 255, 255],
         ...,
         [255, 255, 255,  ..., 255, 255, 255],
         [255, 255, 255,  ..., 255, 255, 255],
         [255, 255, 255,  ..., 255, 255, 255]],

        [[205, 205, 205,  ..., 205, 205, 205],
         [205, 205, 205,  ..., 205, 205, 205],
         [205, 205, 205,  ..., 205, 205, 205],
         ...,
         [205, 205, 205,  ..., 205, 205, 205],
         [205, 205, 205,  ..., 205, 205, 205],
         [205, 205, 205,  ..., 205, 205, 205]],

        [[155, 155, 155,  ..., 155, 155, 155],
         [155, 155, 155,  ..., 155, 155, 155],
         [155, 155, 155,  ..., 155, 155, 155],
         ...,
         [155, 155, 155,  ..., 155, 155, 155],
         [155, 155, 155,  ..., 155, 155, 155],
         [155, 155, 155,  ..., 155, 155, 155]]], dtype=torch.uint8)
```


In [ ]:
import torch as t
import einops

def solve(img):
    """(c, h, w) -> (c, 3h, w): each row appears 3x in a row."""
    return einops.repeat(img, 'c h w -> c (_____) w')


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(317)


In [ ]:
#@title 💡 Solution — Problem 317
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

arr = t.tensor(np.load('/delta_numbers.npy'))
img = arr[0]

def solve(img):
    return einops.repeat(img, 'c h w -> c (h 3) w')

print(solve(img))


<!-- dd:dd-q351 -->

### Problem 351 · guided

Write solve(img) for an (H, W, C) image: double BOTH spatial dimensions by repeating each pixel into a 2×2 block (nearest-neighbor upscaling). Pattern: einops.repeat 'h w c -> (h 2) (w 2) c'.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
tensor([[[255, 205, 155],
         [255, 205, 155],
         [255, 205, 155],
         ...,
         [255, 205, 155],
         [255, 205, 155],
         [255, 205, 155]],

        [[255, 205, 155],
         [255, 205, 155],
         [255, 205, 155],
         ...,
         [255, 205, 155],
         [255, 205, 155],
         [255, 205, 155]],

        [[255, 205, 155],
         [255, 205, 155],
         [255, 205, 155],
         ...,
         [255, 205, 155],
         [255, 205, 155],
         [255, 205, 155]],
… (truncated)
```


<details>
<summary>Hints</summary>

1. Double both spatial dims with each pixel becoming a 2×2 block —
   stretch, and which order makes copies stay WITH their pixel?
2. Factor fast: `(h 2) (w 2)`.
3. `'h w c -> (h 2) (w 2) c'` — check one corner block.

</details>


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img = arr[0]

def solve(img):
    # Write your solution here
    return None

print(solve(img))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(351)


In [ ]:
#@title 💡 Solution — Problem 351
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img = arr[0]

def solve(img):
    return einops.repeat(img, 'h w c -> (h 2) (w 2) c')

print(solve(img))


<!-- dd:dd-q338 -->

### Problem 338 · independent

Write a function solve(cls, steps) that takes a class-embedding tensor of shape (b, d) and an integer steps, and BROADCASTS each embedding across steps time steps: return shape (b, steps, d) where every time slice of item n equals cls[n].

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[1., 2.],
         [1., 2.],
         [1., 2.]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(cls, steps):
    """Return shape (b, steps, d) where every time slice of item n equals cls[n]."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[1.0, 2.0]]), 3))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(338)


In [ ]:
#@title 💡 Solution — Problem 338
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(cls, steps):
    return einops.repeat(cls, 'b d -> b steps d', steps=steps)


print(solve(t.tensor([[1.0, 2.0]]), 3))


<!-- dd:dd-q348 -->

### Problem 348 · independent

Write solve(imgs) for a (B, H, W, C) batch: make each image three times as TALL by repeating each ROW three times consecutively (row 0 three times, then row 1 three times, …). Pattern: einops.repeat 'b h w c -> b (h r) w c' with r=3.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
tensor([[[[255, 205, 155],
          [255, 205, 155],
          [255, 205, 155],
          ...,
          [255, 205, 155],
          [255, 205, 155],
          [255, 205, 155]],

         [[255, 205, 155],
          [255, 205, 155],
          [255, 205, 155],
          ...,
          [255, 205, 155],
          [255, 205, 155],
          [255, 205, 155]],

         [[255, 205, 155],
          [255, 205, 155],
          [255, 205, 155],
          ...,
          [255, 205, 155],
          [255, 205, 155],
          [255, 205, 155]],
… (truncated)
```


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
imgs = arr

def solve(imgs):
    # Write your solution here
    return None

print(solve(imgs))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(348)


In [ ]:
#@title 💡 Solution — Problem 348
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
imgs = arr

def solve(imgs):
    return einops.repeat(imgs, 'b h w c -> b (h r) w c', r=3)

print(solve(imgs))


<!-- dd:dd-q385 -->

### Problem 385 · independent

Write solve(img) for a (C, H, W) image: make it four times as WIDE with each original column appearing four times IN SEQUENCE (column 0 four times, then column 1 four times, …). Pattern: einops.repeat 'c h w -> c h (w 4)'.

**Expected output** — run the cell below once `solve` is right and it should print this. This one draws an image; below is the tensor behind it.

```text
tensor([[[255, 255, 255,  ..., 255, 255, 255],
         [255, 255, 255,  ..., 255, 255, 255],
         [255, 255, 255,  ..., 255, 255, 255],
         ...,
         [255, 255, 255,  ..., 255, 255, 255],
         [255, 255, 255,  ..., 255, 255, 255],
         [255, 255, 255,  ..., 255, 255, 255]],

        [[205, 205, 205,  ..., 205, 205, 205],
         [205, 205, 205,  ..., 205, 205, 205],
         [205, 205, 205,  ..., 205, 205, 205],
         ...,
         [205, 205, 205,  ..., 205, 205, 205],
         [205, 205, 205,  ..., 205, 205, 205],
         [205, 205, 205,  ..., 205, 205, 205]],

        [[155, 155, 155,  ..., 155, 155, 155],
         [155, 155, 155,  ..., 155, 155, 155],
         [155, 155, 155,  ..., 155, 155, 155],
         ...,
         [155, 155, 155,  ..., 155, 155, 155],
         [155, 155, 155,  ..., 155, 155, 155],
         [155, 155, 155,  ..., 155, 155, 155]]], dtype=torch.uint8)
```


In [ ]:
import torch as t
import einops

arr = t.tensor(np.load('/delta_numbers.npy'))
img = arr[0]

def solve(img):
    # Write your solution here
    return None

print(solve(img))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(385)


In [ ]:
#@title 💡 Solution — Problem 385
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

arr = t.tensor(np.load('/delta_numbers.npy'))
img = arr[0]

def solve(img):
    return einops.repeat(img, 'c h w -> c h (w 4)')

print(solve(img))


<!-- dd:dd-q341 -->

### Problem 341 · independent

Write a function solve(img, k) that takes a channels-first image (c, h, w) and returns shape (c, k*h, w) with the WHOLE image stacked k times vertically — copy after copy, via repeat's '(k h)' grouping where the new index varies SLOWEST. (A companion drill stretches rows instead — order inside the parentheses is the whole difference.)

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[0., 1.],
         [2., 3.],
         [0., 1.],
         [2., 3.]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img, k):
    """Return shape (c, k*h, w) with the WHOLE image stacked k times vertically — copy after copy, via repeat's '(k h"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(4.0).reshape(1, 2, 2), 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(341)


In [ ]:
#@title 💡 Solution — Problem 341
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img, k):
    return einops.repeat(img, 'c h w -> c (k h) w', k=k)


print(solve(t.arange(4.0).reshape(1, 2, 2), 2))


<!-- dd:dd-q339 -->

### Problem 339 · independent

Write a function solve(weights, d) that takes per-token scalar weights of shape (b, t) and a feature size d, and EXPANDS them to shape (b, t, d) — each scalar copied across all d feature positions, ready to multiply elementwise against a (b, t, d) activation tensor.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[1., 1.],
         [2., 2.]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(weights, d):
    """Implement the described contraction and return the result."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[1.0, 2.0]]), 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(339)


In [ ]:
#@title 💡 Solution — Problem 339
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(weights, d):
    return einops.repeat(weights, 'b t -> b t d', d=d)


print(solve(t.tensor([[1.0, 2.0]]), 2))


<!-- dd:dd-q383 -->

### Problem 383 · independent

Write a function solve(img, r) that takes a channels-first image (c, h, w) and DUPLICATES the whole channel block r times: return shape (r*c, h, w) via repeat's '(r c)' grouping — channels 0..c-1, then the same again, r blocks total.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[0., 1.],
         [2., 3.]],

        [[0., 1.],
         [2., 3.]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img, r):
    """Return shape (r*c, h, w) via repeat's '(r c)' grouping — channels 0..c-1, then the same again, r blocks total."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(4.0).reshape(1, 2, 2), 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(383)


In [ ]:
#@title 💡 Solution — Problem 383
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img, r):
    return einops.repeat(img, 'c h w -> (r c) h w', r=r)


print(solve(t.arange(4.0).reshape(1, 2, 2), 2))


<!-- dd:dd-q352 -->

### Problem 352 · independent

Write a function solve(img, k) that takes a channels-first image (c, h, w) and STRETCHES it vertically by k: return shape (c, h*k, w) with each ROW duplicated k times in place via '(h k)' — the new index varying FASTEST. (The companion tiling drill uses '(k h)' — copies of the whole image instead.)

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[0., 1.],
         [0., 1.],
         [2., 3.],
         [2., 3.]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img, k):
    """Return shape (c, h*k, w) with each ROW duplicated k times in place via '(h k)' — the new index varying FASTEST"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(4.0).reshape(1, 2, 2), 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(352)


In [ ]:
#@title 💡 Solution — Problem 352
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img, k):
    return einops.repeat(img, 'c h w -> c (h k) w', k=k)


print(solve(t.arange(4.0).reshape(1, 2, 2), 2))


<!-- dd:dd-q355 -->

### Problem 355 · independent

Write a function solve(arr, k) that takes a channels-first batch (b, c, h, w) with b >= 2, keeps the FIRST TWO images, stacks them vertically, and repeats the whole strip k times horizontally: return shape (c, 2*h, k*w) via einops.repeat(arr[:2], 'b c h w -> c (b h) (k w)', k=k).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[0., 1., 0., 1.],
         [2., 3., 2., 3.],
         [4., 5., 4., 5.],
         [6., 7., 6., 7.]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr, k):
    """Return shape (c, 2*h, k*w) via einops.repeat(arr[:2], 'b c h w -> c (b h) (k w)', k=k)."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(2, 1, 2, 2), 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(355)


In [ ]:
#@title 💡 Solution — Problem 355
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr, k):
    return einops.repeat(arr[:2], 'b c h w -> c (b h) (k w)', k=k)


print(solve(t.arange(8.0).reshape(2, 1, 2, 2), 2))


#### Common mistakes

- **"repeat is rearrange with bigger numbers."** — rearrange is a
  bijection (every input element appears once); repeat COPIES. The
  found-on-the-right-only names/factors are what license the copying.
- **"'(h 2)' and '(2 h)' both just double the axis."** — Same length,
  different layout: h-slow keeps copies adjacent to their source row
  (stretch); 2-slow lays whole blocks end to end (tile). Tasks say
  "consecutively"/"each row" vs "whole image again" — map the words to
  the order.
- **"Broadcasting makes repeat unnecessary."** — Broadcasting stretches
  virtually within an operation; repeat materializes an actual tensor with
  the new shape — which is what a drill's return-shape contract (and much
  downstream code) requires. Know which one you're producing.


<!-- dd:dd-kp-einops-dl-flatten-heads -->

## Deep-learning shapes — flattening and attention heads

`einops.dl-flatten-heads`


Two shape manoeuvres appear in virtually every model's forward pass, and
both are single einops patterns:

**The classifier flatten.** Convolutional features (b, c, h, w) must become
(b, c·h·w) before a linear layer:

> `'b c h w -> b (c h w)'`

— batch stays, everything else merges (c slow: all of channel 0's pixels,
then channel 1's — the order PyTorch's `.view(b, -1)` produces, so weights
transfer). A single image entering a batch-expecting classifier combines
this with the singleton trick: `'c h w -> 1 (c h w)'`. The full-tensor
collapse `'b c h w -> b'`-style sums are reduce; TOTAL flattening to a
scalar count of axes is `'... -> (...)'`-shaped merges — same grammar
throughout.

**The attention-head split/merge.** Multi-head attention stores per-head
outputs as (b, nh, t, d); tokens want them CONCATENATED as (b, t, nh·d):

> merge: `'b nh t d -> b t (nh d)'`
> — head index slow: head 0's d features first within each token vector.
> split (the inverse, entering attention): `'b t (nh d) -> b nh t d', nh=N`
> — the packed feature axis DECLARES its (heads × per-head) structure.

These two patterns are why einops is beloved in transformer code: the
head-packing convention (nh slow) is load-bearing — get it wrong and
weights trained with one convention silently misread activations from the
other — and the pattern states it where a reshape hides it.

Everything here is the merge/split KPs at model-shaped rank; what this KP
adds is fluency with the CONVENTIONS (batch first, which name is slow) that
the drills — and real checkpoints — assume.


Task: flatten conv features for a classifier; merge attention heads; split
them back.


In [ ]:
import torch as t
import einops

feats = t.arange(24.0).reshape(2, 3, 2, 2)     # (b, c, h, w)

# Classifier flatten: batch survives, (c h w) merge in that order.
flat = einops.rearrange(feats, 'b c h w -> b (c h w)')
assert flat.shape == (2, 12)
# c slow: the first 4 entries of item 0 are channel 0's pixels.
assert flat[0, :4].tolist() == feats[0, 0].ravel().tolist()

# Heads merge: (b, nh, t, d) -> (b, t, (nh d)).
heads = t.arange(16.0).reshape(1, 2, 2, 4)     # (b, nh=2, t, d=4)
merged = einops.rearrange(heads, 'b nh t d -> b t (nh d)')
assert merged.shape == (1, 2, 8)
# Token 0's vector = head 0's features then head 1's (nh slow):
assert merged[0, 0].tolist() == (heads[0, 0, 0].tolist()
                                 + heads[0, 1, 0].tolist())

# The inverse split — declare how the packed axis factors.
unmerged = einops.rearrange(merged, 'b t (nh d) -> b nh t d', nh=2)
assert t.equal(unmerged, heads)          # round trip exact
print("feats", tuple(feats.shape), "-> flat", tuple(flat.shape))
print("item 0 starts with channel 0's pixels:", flat[0, :4])
print("heads", tuple(heads.shape), "-> merged", tuple(merged.shape))
print("token 0 =", merged[0, 0], " (head 0's four, then head 1's)")
print("split back exactly:", bool(t.equal(unmerged, heads)))




Why each step:

1. The `flat[0, :4]` check pins the merge order to something inspectable —
   and matches the framework convention, which is the practical point:
   these patterns interoperate with real model weights.
2. The token-vector concatenation check reads the head-packing convention
   off actual data: head 0 first. When a drill (or paper) says "heads
   concatenated per token", this is the layout it means.
3. The round trip closes the loop: merge and split with matching
   conventions are exact inverses. If your split of someone else's packed
   tensor doesn't round-trip through their merge, your nh-slow assumption
   is wrong — debug the CONVENTION, not the code.


<!-- dd:dd-q356 -->

### Problem 356 · faded — your turn

Flatten each image of a batch for a linear layer.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0., 1., 2., 3.],
        [4., 5., 6., 7.]])
```


In [ ]:
import torch as t
import einops

def solve(x):
    """(b, c, h, w) -> (b, c*h*w), standard framework order."""
    return einops.rearrange(x, '_____')


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(356)


In [ ]:
#@title 💡 Solution — Problem 356
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x):
    return einops.rearrange(x, 'b c h w -> b (c h w)')


print(solve(t.arange(8.0).reshape(2, 1, 2, 2)))


<!-- dd:dd-q396 -->

### Problem 396 · guided

Write a function solve(seq, nh) that takes a transformer sequence of shape (b, t, nh*d) — the head outputs concatenated per token, head index slowest — and SPLITS the heads back out: return shape (b, nh, t, d) via 'b t (nh d) -> b nh t d'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[0., 1.],
          [4., 5.]],

         [[2., 3.],
          [6., 7.]]]])
```


<details>
<summary>Hints</summary>

1. The packed sequence (b, t, nh·d), head index slowest, must SPLIT back
   into heads — inverse of the merge you just did.
2. The parens move to the input side; one keyword names the head count.
3. `'b t (nh d) -> b nh t d', nh=nh` — round-trip it against a merge to be
   sure.

</details>


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(seq, nh):
    """Return shape (b, nh, t, d) via 'b t (nh d) -> b nh t d'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(1, 2, 4), 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(396)


In [ ]:
#@title 💡 Solution — Problem 396
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(seq, nh):
    return einops.rearrange(seq, 'b t (nh d) -> b nh t d', nh=nh)


print(solve(t.arange(8.0).reshape(1, 2, 4), 2))


<!-- dd:dd-q384 -->

### Problem 384 · independent

Write a function solve(x_heads) that takes multi-head attention output of shape (b, nh, t, d) and CONCATENATES the heads back together per token: return shape (b, t, nh*d) via 'b nh t d -> b t (nh d)'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[0., 1., 4., 5.],
         [2., 3., 6., 7.]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x_heads):
    """Return shape (b, t, nh*d) via 'b nh t d -> b t (nh d)'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(1, 2, 2, 2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(384)


In [ ]:
#@title 💡 Solution — Problem 384
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x_heads):
    return einops.rearrange(x_heads, 'b nh t d -> b t (nh d)')


print(solve(t.arange(8.0).reshape(1, 2, 2, 2)))


<!-- dd:dd-q394 -->

### Problem 394 · independent

Write a function solve(img) that takes a single channels-first image (c, h, w), treats it as a batch of ONE, and returns the flattened prediction-style vector batch of shape (1, c*h*w): add the leading batch axis then flatten the rest — the two rearranges compose into 'c h w -> () (c h w)'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0., 1., 2., 3.]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img):
    """Return the flattened prediction-style vector batch of shape (1, c*h*w)."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(4.0).reshape(1, 2, 2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(394)


In [ ]:
#@title 💡 Solution — Problem 394
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img):
    return einops.rearrange(img, 'c h w -> () (c h w)')


print(solve(t.arange(4.0).reshape(1, 2, 2)))


<!-- dd:dd-q349 -->

### Problem 349 · independent

Write a function solve(x) that takes a 4-D tensor (b, c, h, w) and returns the SUM of every element as a 0-D result — einops.reduce with the empty output pattern '->'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor(1536.)
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x):
    """Return the SUM of every element as a 0-D result — einops.reduce with the empty output pattern '->'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((2, 12, 8, 8))))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(349)


In [ ]:
#@title 💡 Solution — Problem 349
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x):
    return einops.reduce(x, 'b c h w -> ', 'sum')


print(solve(t.ones((2, 12, 8, 8))))


#### Common mistakes

- **"Any flatten order works if shapes match."** — The linear layer's
  weights were trained against ONE packing order. (c h w) vs (h w c)
  flattens produce equal shapes and incompatible semantics. Follow the
  framework convention the drill names.
- **"Head merge/split order is a style choice."** — nh-slow is the
  transformer ecosystem's convention; violating it silently permutes
  features. The einops pattern documents the choice — that's half its
  value.
- **"These need reshape + transpose + reshape chains."** — Each is ONE
  pattern. If your solution has intermediate .transpose calls around
  reshapes, the pattern can absorb them (and check the shapes while it's
  at it).


<!-- dd:dd-kp-einops-channel-groups-temporal -->

## Channel groups and temporal windows

`einops.channel-groups-temporal`


The capstone patterns: channel axes and time axes that carry HIDDEN
STRUCTURE, unpacked and repacked with everything this course built. Two
families:

**Grouped channels.** A channel axis of size g·c often means "g groups of c
channels" — and the single load-bearing question is **which index is slow**:

- **Group SLOWEST** (`(g c)`): channels are g0c0, g0c1, …, g1c0… — whole
  groups in blocks. Splitting groups out: `'b (g c) h w -> g b c h w', g=G`.
- **Group FASTEST** (`(c g)`): channels INTERLEAVE — g0c0, g1c0, g2c0,
  g0c1… Splitting the same tensor: `'b (c g) h w -> g b c h w', g=G`.

Identical shapes, opposite layouts. The drills alternate between the two
precisely to force the read: find the task's phrase ("group index
slowest" / "channels interleave") and place the factor accordingly.
Group operations then compose: swap two group factors
(`'b (g1 g2 c) h w -> b (g2 g1 c) h w'`), transpose groups against
within-group channels (`(g c) -> (c g)`) — all one-pattern moves.

**Temporal windows.** Sequences (b, c, t) pool and chunk exactly like
spatial axes: average non-overlapping PAIRS of time steps —
`reduce('b c (t two) -> b c t', 'mean', two=2)`; length-w windows —
`two→w`. Multi-head sequence packing, chunked attention, and space-time
tensors are these same factored axes on the time dimension.

Nothing in this KP is new grammar — it is fluency under pressure: rank-4/5
tensors, two conventions in play, and the pattern as the single place the
truth lives. This is the level ARENA's einops exercises (and real model
surgery) operate at.


Task: split interleaved channel groups; swap group blocks; average-pool
time pairs.


In [ ]:
import torch as t
import einops

# 6 channels = 3 channels x 2 groups, GROUP INDEX FASTEST (interleaved):
# channel order is g0c0, g1c0, g0c1, g1c1, g0c2, g1c2.
x = t.arange(12.0).reshape(1, 6, 1, 2)          # (b, 6, h, w)

split = einops.rearrange(x, 'b (c g) h w -> g b c h w', g=2)
assert split.shape == (2, 1, 3, 1, 2)
# Group 0 must hold original channels 0, 2, 4 (every second one):
assert t.equal(split[0, 0, :, 0, 0], x[0, ::2, 0, 0])

# Same data read as GROUP SLOWEST would give a different (wrong here) split:
wrong = einops.rearrange(x, 'b (g c) h w -> g b c h w', g=2)
assert not t.equal(split, wrong)          # conventions matter!

# Temporal pooling: average adjacent pairs of time steps.
seq = t.arange(8.0).reshape(1, 1, 8)            # (b, c, t=8)
halved = einops.reduce(seq, 'b c (t two) -> b c t', 'mean', two=2)
assert halved[0, 0].tolist() == [0.5, 2.5, 4.5, 6.5]
print("channels        ", x[0, :, 0, 0])
print("'(c g)' group 0 ", split[0, 0, :, 0, 0], " <- every second channel")
print("'(g c)' group 0 ", wrong[0, 0, :, 0, 0], " <- a contiguous half")
print("seq", seq[0, 0], "-> pairwise mean", halved[0, 0])




Why each step:

1. The interleaved split's check (`group 0 == channels 0, 2, 4`) is the
   ground truth for "(c g) with g fast": stride-2 selection. If the task
   had said "group index slowest", group 0 would be channels 0, 1, 2 —
   the `wrong` variant. One assert distinguishes them.
2. Running BOTH readings on the same data — and seeing them differ — is
   the inoculation this KP exists for: shape-compatible ≠ correct, and
   only the task's layout sentence decides.
3. The temporal pooling is spatial pooling with the axis renamed — placed
   here to make the transfer explicit: axes are axes; the pattern language
   doesn't care whether they're pixels or time steps.


<!-- dd:dd-q362 -->

### Problem 362 · faded — your turn

Split out interleaved channel groups (group index FASTEST).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[[0., 1.]],

          [[4., 5.]]]],



        [[[[2., 3.]],

          [[6., 7.]]]]])
```


In [ ]:
import torch as t
import einops

def solve(x, split):
    """(b, c*split, h, w), groups interleaved -> (split, b, c, h, w)."""
    return einops.rearrange(x, 'b (_____) h w -> split b c h w', split=split)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(362)


In [ ]:
#@title 💡 Solution — Problem 362
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x, split):
    return einops.rearrange(x, 'b (c split) h w -> split b c h w', split=split)


print(solve(t.arange(8.0).reshape(1, 4, 1, 2), 2))


<!-- dd:dd-q375 -->

### Problem 375 · guided

Write a function solve(x_seq) that takes a temporal tensor of shape (b, c, 2*t) and DOWNSAMPLES time by averaging non-overlapping PAIRS: return shape (b, c, t) via 'b c (t two) -> b c t' with 'mean'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[0.5000, 2.5000, 4.5000, 6.5000]]])
```


<details>
<summary>Hints</summary>

1. (b, c, 2t) downsampled by averaging non-overlapping PAIRS — temporal
   pooling with window 2.
2. The time axis factors as (t × 2); the window name reduces away.
3. `einops.reduce(x_seq, 'b c (t two) -> b c t', 'mean', two=2)`.

</details>


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x_seq):
    """Return shape (b, c, t) via 'b c (t two) -> b c t' with 'mean'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(1, 1, 8)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(375)


In [ ]:
#@title 💡 Solution — Problem 375
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x_seq):
    return einops.reduce(x_seq, 'b c (t two) -> b c t', 'mean', two=2)


print(solve(t.arange(8.0).reshape(1, 1, 8)))


<!-- dd:dd-q369 -->

### Problem 369 · independent

Write a function solve(x, g1, g2) that takes a tensor of shape (b, g1*g2*c, h, w) whose channel axis factors as (g1, g2, c) with g1 slowest, and SWAPS the two group levels: return the same shape with channels reordered as (g2, g1, c) — 'b (g1 g2 c) h w -> b (g2 g1 c) h w'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[0., 1.]],

         [[4., 5.]],

         [[2., 3.]],

         [[6., 7.]]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x, g1, g2):
    """Return the same shape with channels reordered as (g2, g1, c) — 'b (g1 g2 c) h w -> b (g2 g1 c) h w'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(1, 4, 1, 2), 2, 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(369)


In [ ]:
#@title 💡 Solution — Problem 369
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x, g1, g2):
    return einops.rearrange(x, 'b (g1 g2 c) h w -> b (g2 g1 c) h w', g1=g1, g2=g2)


print(solve(t.arange(8.0).reshape(1, 4, 1, 2), 2, 2))


<!-- dd:dd-q372 -->

### Problem 372 · independent

Write a function solve(x, g) that takes a tensor of shape (b, g*c, h, w) with g channel groups (group index slowest) and TRANSPOSES groups against within-group channels: return the same shape reordered by 'b (g c) h w -> b (c g) h w' — after which consecutive channels come from DIFFERENT groups.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[0.]],

         [[2.]],

         [[1.]],

         [[3.]]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x, g):
    """Return the same shape reordered by 'b (g c) h w -> b (c g) h w' — after which consecutive channels come from D"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(4.0).reshape(1, 4, 1, 1), 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(372)


In [ ]:
#@title 💡 Solution — Problem 372
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x, g):
    return einops.rearrange(x, 'b (g c) h w -> b (c g) h w', g=g)


print(solve(t.arange(4.0).reshape(1, 4, 1, 1), 2))


<!-- dd:dd-q378 -->

### Problem 378 · independent

Write a function solve(x_seq, w) that takes a temporal tensor (b, c, t*w) and average-pools time with non-overlapping windows of length w: return shape (b, c, t) via 'b c (t w) -> b c t' with 'mean'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[1.5000, 5.5000]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x_seq, w):
    """Return shape (b, c, t) via 'b c (t w) -> b c t' with 'mean'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(1, 1, 8), 4))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(378)


In [ ]:
#@title 💡 Solution — Problem 378
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x_seq, w):
    return einops.reduce(x_seq, 'b c (t w) -> b c t', 'mean', w=w)


print(solve(t.arange(8.0).reshape(1, 1, 8), 4))


<!-- dd:dd-q397 -->

### Problem 397 · independent

Write a function solve(arr) that takes a batch (b, c, h, w) and returns shape (h, w, b*c): spatial axes leading, batch and channels merged (batch slowest) at the END — 'b c h w -> h w (b c)'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[0., 4.],
         [1., 5.]],

        [[2., 6.],
         [3., 7.]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Return shape (h, w, b*c)."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(2, 1, 2, 2)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(397)


In [ ]:
#@title 💡 Solution — Problem 397
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    return einops.rearrange(arr, 'b c h w -> h w (b c)')


print(solve(t.arange(8.0).reshape(2, 1, 2, 2)))


<!-- dd:dd-q316 -->

### Problem 316 · independent

Write a function solve(x, coord) that takes a tensor of shape (b, coord*k, h, w) whose channel axis interleaves `coord` groups of k channels (group index varying SLOWEST), and returns shape (coord, b, k, h, w) with the groups split out and moved to the front — 'b (coord k) h w -> coord b k h w'.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[[0., 1.]]]],



        [[[[2., 3.]]]],



        [[[[4., 5.]]]],



        [[[[6., 7.]]]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x, coord):
    """Return shape (coord, b, k, h, w) with the groups split out and moved to the front — 'b (coord k) h w -> coord """
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(1, 4, 1, 2), 4))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(316)


In [ ]:
#@title 💡 Solution — Problem 316
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x, coord):
    return einops.rearrange(x, 'b (coord k) h w -> coord b k h w', coord=coord)


print(solve(t.arange(8.0).reshape(1, 4, 1, 2), 4))


<!-- dd:dd-q359 -->

### Problem 359 · independent

Write a function solve(x, split) that takes a tensor of shape (b, split*c, h, w) whose channel axis holds `split` consecutive groups (group index SLOWEST) and returns shape (split, b, c, h, w) — the groups separated and moved to the front, ready to unpack as part1, part2, ... = solve(x, 2).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[[[0., 1.]],

          [[2., 3.]]]],



        [[[[4., 5.]],

          [[6., 7.]]]]])
```


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x, split):
    """Return shape (split, b, c, h, w) — the groups separated and moved to the front, ready to unpack as part1, part"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(1, 4, 1, 2), 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(359)


In [ ]:
#@title 💡 Solution — Problem 359
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x, split):
    return einops.rearrange(x, 'b (split c) h w -> split b c h w', split=split)


print(solve(t.arange(8.0).reshape(1, 4, 1, 2), 2))


#### Common mistakes

- **"(g c) and (c g) differ only in variable naming."** — They are
  opposite memory layouts: blocks vs interleave. Same shape, different
  tensor. The task's "slowest/fastest/interleaved" sentence is the spec —
  translate it to paren order before writing anything.
- **"Group tricks need index arithmetic over channel numbers."** — Every
  grouped-channel task in the bank is one rearrange with the right
  factoring. Hand-computed channel indices are the sign you're fighting
  the notation.
- **"Time axes are special."** — To the pattern language a time axis is an
  axis. Pooling, chunking, windowing transfer verbatim from the spatial
  versions — reuse the pattern, rename the letters.
